# INSTALLS

In [ ]:
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl

# IMPORTS

In [ ]:
import torch, random
import numpy as np
import pandas as pd
import torch.nn as nn

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter


# DEVICE

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# PATHS

In [ ]:
TRAIN_SEQ = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv"
TEST_SEQ  = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"

# HYPERPARAMS

In [ ]:
WINDOW_SIZE = 400
STRIDE = 200
K_NEIGHBORS = 8

# SEQUENCE UTILS

In [ ]:
NUC_MAP = {'A':0,'U':1,'G':2,'C':3}

def clean_sequence(seq):
    return "".join([s for s in seq.upper() if s in NUC_MAP])

def one_hot(seq):
    x = torch.zeros(len(seq),4)
    for i,s in enumerate(seq):
        x[i,NUC_MAP[s]] = 1
    return x

# BASE-PAIR RULES (BIOLOGY)

In [ ]:
def is_base_pair(a, b):
    return (
        (a=='A' and b=='U') or
        (a=='U' and b=='A') or
        (a=='G' and b=='C') or
        (a=='C' and b=='G') or
        (a=='G' and b=='U') or   # wobble
        (a=='U' and b=='G')
    )

# GRAPH BUILDER (PRODUCTION)

In [ ]:
def build_graph(seq, x, coords=None):
    L = x.size(0)

    row, col = [], []
    edge_type = []  # 0 local, 1 global, 2 base-pair

    for i in range(L):

        # local edges
        for j in range(max(0, i-K_NEIGHBORS), min(L, i+K_NEIGHBORS+1)):
            if i == j: continue
            row.append(i); col.append(j); edge_type.append(0)

        # global edges
        for j in random.sample(range(L), min(K_NEIGHBORS, L)):
            if i == j: continue
            row.append(i); col.append(j); edge_type.append(1)

        # base-pair edges (global scan)
        for j in range(L):
            if abs(i-j) > 3 and is_base_pair(seq[i], seq[j]):
                row.append(i); col.append(j); edge_type.append(2)

    edge_index = torch.tensor([row,col], dtype=torch.long)

    rel = (edge_index[0]-edge_index[1]).float().unsqueeze(1)/L
    dist = rel.abs()

    edge_type = torch.tensor(edge_type).unsqueeze(1).float()

    edge_attr = torch.cat([dist, rel, edge_type], dim=1)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    if coords is not None:
        data.y = coords

    # stable init (important for TM-score)
    data.pos = torch.randn(L,3) * 0.01

    return data

# DATASET

In [ ]:
class RNAWindowDataset(Dataset):
    def __init__(self, seq_csv, label_csv=None):
        self.df = pd.read_csv(seq_csv)
        self.has_labels = label_csv is not None

        self.seq_map = {
            row["target_id"]: clean_sequence(row["sequence"])
            for _,row in self.df.iterrows()
        }

        if self.has_labels:
            labels = pd.read_csv(label_csv, low_memory=False)
            labels["sid"] = labels["ID"].str.split("_").str[0]
            labels["idx"] = labels["ID"].str.split("_").str[1].astype(int)

            self.coords = {}
            for k,g in labels.groupby("sid"):
                g = g.sort_values("idx")
                xyz = torch.tensor(g[["x_1","y_1","z_1"]].values, dtype=torch.float32)

                valid = ~torch.isnan(xyz).any(dim=1)
                xyz = xyz[valid]

                if len(xyz)>0:
                    self.coords[k] = xyz

        self.samples = []

        for sid in self.df["target_id"]:
            seq = self.seq_map[sid]
            L = len(seq)

            if self.has_labels and sid in self.coords:
                L = min(L, self.coords[sid].shape[0])

            for s in range(0, L, STRIDE):
                e = min(s+WINDOW_SIZE, L)
                if e-s >= 20:
                    self.samples.append((sid,s,e))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sid,s,e = self.samples[idx]
        seq = self.seq_map[sid][s:e]

        coords = None
        if self.has_labels and sid in self.coords:
            coords = self.coords[sid][s:e]

        x = one_hot(seq)
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        return build_graph(seq, x, coords)

# MODEL (EDGE-TYPE AWARE EGNN)

In [ ]:
class EGNNLayer(nn.Module):
    def __init__(self, hidden):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden*2 + 3, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self,x,pos,edge_index,edge_attr):
        row,col = edge_index
        rel = pos[row]-pos[col]

        m = self.edge_mlp(torch.cat([x[row],x[col],edge_attr],dim=1))

        agg = scatter(m,row,dim=0,dim_size=x.size(0),reduce="mean")
        x = self.node_mlp(torch.cat([x,agg],dim=1))

        trans = self.coord_mlp(m) * rel
        delta = scatter(trans,row,dim=0,dim_size=pos.size(0),reduce="mean")

        pos = pos + 0.5 * delta  # 🔥 controlled update (important)

        return x,pos


class EGNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(5,128)
        self.layers = nn.ModuleList([EGNNLayer(128) for _ in range(6)])

    def forward(self,data):
        x,pos = data.x,data.pos
        x = self.emb(x)

        for l in self.layers:
            x,pos = l(x,pos,data.edge_index,data.edge_attr)

        return pos

# LOSS (TM-STABLE)

In [ ]:
def hybrid_loss(pred, target, batch):
    loss = 0
    n = batch.max()+1

    for i in range(n):
        mask = batch==i
        P = pred[mask]
        Q = target[mask]

        Pc = P - P.mean(0,keepdim=True)
        Qc = Q - Q.mean(0,keepdim=True)

        scale = torch.sqrt((Qc**2).sum(dim=1).mean() + 1e-6)

        Pn = Pc / scale
        Qn = Qc / scale

        mse = ((Pn-Qn)**2).mean()

        dist_loss = ((torch.cdist(Pn,Pn)-torch.cdist(Qn,Qn))**2).mean()

        L = P.shape[0]
        d0 = 1.24*(L-15)**(1/3)-1.8 if L>=30 else 0.5
        d = torch.norm(Pn-Qn, dim=1)
        tm = (1/(1+(d/d0)**2)).mean()

        loss += 0.3*mse + 0.3*dist_loss + 0.4*(1-tm)

    return loss/n

# TRAIN

In [ ]:
train_ds = RNAWindowDataset(TRAIN_SEQ, TRAIN_LBL)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

model = EGNNModel().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

for e in range(5):
    model.train()
    total = 0

    for data in train_loader:
        data = data.to(DEVICE)

        opt.zero_grad()
        pred = model(data)
        loss = hybrid_loss(pred, data.y, data.batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        total += loss.item()

    print(f"Epoch {e} | loss = {total/len(train_loader):.6f}")

# INFERENCE + SUBMISSION (5 STRUCTURES)

In [ ]:
test_df = pd.read_csv(TEST_SEQ)
dataset = RNAWindowDataset(TEST_SEQ, None)

def predict_full(model, sid, num_samples=5):
    seq = dataset.seq_map[sid]
    L = len(seq)

    outputs = []

    for _ in range(num_samples):
        coord_sum = torch.zeros(L,3)
        count = torch.zeros(L,1)

        for (s_id,s,e) in dataset.samples:
            if s_id != sid: continue

            chunk = seq[s:e]

            x = one_hot(chunk)
            pos_feat = torch.arange(len(chunk)).float().unsqueeze(-1)/len(chunk)
            x = torch.cat([x,pos_feat],dim=1)

            data = build_graph(chunk, x).to(DEVICE)

            with torch.no_grad():
                pred = model(data).cpu()

            coord_sum[s:e] += pred
            count[s:e] += 1

        outputs.append((coord_sum/(count+1e-6)).numpy())

    return outputs


rows = []

for _, row in test_df.iterrows():
    sid = row["target_id"]
    seq = clean_sequence(row["sequence"])

    preds = predict_full(model, sid)

    for i,res in enumerate(seq):
        out = {
            "ID": f"{sid}_{i+1}",
            "resname": res,
            "resid": i+1
        }

        for k in range(5):
            x,y,z = preds[k][i]
            out[f"x_{k+1}"] = x
            out[f"y_{k+1}"] = y
            out[f"z_{k+1}"] = z

        rows.append(out)

submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

print("✅ submission ready")